# 05 继续 DPO 训练：从 preference 数据到安全验收

你现在已经知道怎么评估 adapter。

今天只解决一个问题：

> 如果还要继续 DPO，应该从哪里进、看哪些文件、怎么避免盲目加 step？

这一页只看 5 个真实入口：

```text
data/processed/dpo_larger_v6_train.jsonl
configs/dpo_qwen05b_v6_naive.yaml
configs/dpo_qwen05b_v7_stage5h.yaml
scripts/train_dpo.py
reports/stage5j_to_5p_prompt7_repair_report.md
```

核心地图是：

```text
DPO JSONL: prompt / chosen / rejected
  -> config 指定起点 adapter、数据、输出目录、显存参数
  -> train_dpo.py 加载 policy model 和 reference model
  -> DPOTrainer.train()
  -> 输出新的 DPO LoRA adapter
  -> 立刻回到 04 的行为门禁验收
```

继续 DPO 的 20% 核心不是公式，而是：数据对、起点、reference、显存参数、验收门禁。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/codex备份/qwen-local-lora-sft-dpo学习版")

print("项目根目录:", PROJECT_ROOT)

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits[:5]:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 0. 今天只看哪些真实项目文件？

| 类型 | 路径 | 今天看它干嘛 |
|---|---|---|
| DPO 数据 | `data/processed/dpo_larger_v6_train.jsonl` | 看 prompt/chosen/rejected 长什么样 |
| v6 配置 | `configs/dpo_qwen05b_v6_naive.yaml` | 看一次成功跑通的 larger DPO 设置 |
| v7 配置 | `configs/dpo_qwen05b_v7_stage5h.yaml` | 看 prompt 7 修复数据怎么接入 |
| DPO 主脚本 | `scripts/train_dpo.py` | 看 policy/reference/DPOTrainer 主链路 |
| 修复复盘 | `reports/stage5j_to_5p_prompt7_repair_report.md` | 看为什么不要继续盲目加训练 |

先只把这 5 个位置连起来，就已经抓住继续 DPO 的主线了。

In [ ]:
for rel in [
    "data/processed/dpo_larger_v6_train.jsonl",
    "configs/dpo_qwen05b_v6_naive.yaml",
    "configs/dpo_qwen05b_v7_stage5h.yaml",
    "scripts/train_dpo.py",
    "reports/stage5j_to_5p_prompt7_repair_report.md",
]:
    print(f"{rel:65s}", "OK" if (PROJECT_ROOT / rel).exists() else "MISSING")

## 1. 先看一条 DPO 数据

SFT 数据是 `messages=[system,user,assistant]`。

DPO 数据更像选择题：

```text
prompt: 同一个问题
chosen: 更希望模型学会的回答
rejected: 更希望模型远离的回答
```

In [ ]:
for rel in [
    "data/processed/dpo_tiny_train.jsonl",
    "data/processed/dpo_larger_v6_train.jsonl",
    "data/processed/dpo_stage5h_prompt7_train.jsonl",
    "data/processed/dpo_stage5m_exact_prompt7_train.jsonl",
]:
    if (PROJECT_ROOT / rel).exists():
        print(f"{rel:55s}", count_jsonl(rel), "rows")

row = read_jsonl("data/processed/dpo_larger_v6_train.jsonl", n=1)[0]
print(json.dumps(row, ensure_ascii=False, indent=2)[:2500])

DPO 的本质可以先背成一句话：

> 对同一个 prompt，让当前模型更偏向 chosen，同时相对远离 rejected。

它不是重新教所有知识，而是在已有 SFT adapter 基础上调整偏好。

## 2. config 是继续 DPO 的控制台

先看 v6 配置。它是本项目目前最好的 DPO 实验 artifact，但不是默认推荐 checkpoint。

In [ ]:
show_file("configs/dpo_qwen05b_v6_naive.yaml", start=1, end=40)

这份配置里最重要的 8 个字段：

| 字段 | 你要理解什么 |
|---|---|
| `model_name` | 底座模型，仍然是 Qwen2.5-0.5B-Instruct |
| `sft_adapter_path` | 继续 DPO 的起点 adapter |
| `dpo_file` | 训练 preference 数据 |
| `eval_file` | preference eval 数据，不等于最终行为验收 |
| `output_dir` | 新 adapter 保存到哪里 |
| `max_length` / `max_prompt_length` | 显存风险的主要旋钮 |
| `learning_rate` / `beta` | DPO 更新力度 |
| `separate_ref_model` | 是否加载单独 frozen reference model |

## 3. train_dpo.py 的代码地图

不要从第一行读到最后一行。先按函数地图读：

| 代码位置 | 你要看什么 | 输入 | 输出 |
|---|---|---|---|
| `read_simple_yaml` | 读 config | yaml 路径 | dict |
| `merged_config` | 合并默认值和命令行 | args + config | 最终参数 |
| `read_dpo_rows` | 读 preference JSONL | prompt/chosen/rejected | Dataset rows |
| `load_policy_model` | 加载可训练模型 | base + adapter | trainable policy |
| `load_reference_model` | 加载冻结参考模型 | base + adapter | frozen reference |
| `DPOConfig` | 训练参数 | config | training_args |
| `DPOTrainer` | 真正训练 | model/ref/data | 新 adapter |

In [ ]:
for keyword in [
    "def read_simple_yaml",
    "def merged_config",
    "def read_dpo_rows",
    "def load_policy_model",
    "def load_reference_model",
    "DPOConfig(",
    "DPOTrainer(",
]:
    find_lines("scripts/train_dpo.py", keyword, context=5)

## 4. policy model 和 reference model 是什么？

DPO 里你要先分清两份模型：

```text
policy model
  当前要训练的模型
  base + sft_adapter_path
  is_trainable=True

reference model
  用来当对照的冻结模型
  base + 同一个起点 adapter
  is_trainable=False
```

粗略理解：policy 可以动，reference 不动。DPO 不希望模型为了追 chosen 把原来的能力乱改。

In [ ]:
find_lines("scripts/train_dpo.py", "is_trainable=True", context=10)
find_lines("scripts/train_dpo.py", "is_trainable=False", context=10)
find_lines("scripts/train_dpo.py", "requires_grad_(False)", context=6)
find_lines("scripts/train_dpo.py", "separate_ref_model", context=10)

## 5. 安全打印训练命令：默认不训练

下面只打印命令，不训练。

如果你真的要跑，把 `RUN_DPO_TRAIN=True`。这会占用显存，并写入 `outputs/`。

In [ ]:
RUN_DPO_TRAIN = False

dpo_cmd = [
    sys.executable, "scripts/train_dpo.py",
    "--config", "configs/dpo_qwen05b_v6_naive.yaml",
    "--local_files_only",
]

print(" ".join(dpo_cmd))

if RUN_DPO_TRAIN:
    result = subprocess.run(dpo_cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, encoding="utf-8", errors="replace")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("默认不运行。先理解：这一步会从 SFT adapter 继续训练出 DPO adapter。")

## 6. 如果要继续，不要只复制 v6

v6 证明硬件能跑，也证明 separate reference 有价值。

但 v6 没有通过最终行为门禁，所以继续训练时最重要的问题不是“再跑多少步”，而是：

```text
我要修哪个失败行为？
有没有 held-out 改写题？
会不会修了 prompt 7，却让旧 prompt 回归？
新 adapter 准备用什么门禁验收？
```

In [ ]:
show_file("configs/dpo_qwen05b_v7_stage5h.yaml", start=1, end=40)
print()
show_file("configs/dpo_qwen05b_v8_stage5m_from_v6.yaml", start=1, end=40)

v7 和 v8 的价值是学习“怎么继续”，不是照抄成最终答案：

- v7 从 SFT v3 出发，用 Stage 5H prompt 7 修复数据。
- v8 从 DPO v6 出发，用更精确的 prompt 7 failure pair。
- 两者 preference accuracy 都很好，但行为门禁仍然没有完整通过。

这就是为什么继续 DPO 必须和评估门禁绑在一起。

## 7. 看最终复盘：为什么停在这里？

In [ ]:
show_file("reports/stage5j_to_5p_prompt7_repair_report.md", start=1, end=110)

你要特别记住这个项目判断：

```text
推荐 checkpoint: outputs/sft_lora_qwen05b_custom_v3_from_v1_patch
最好 DPO artifact: outputs/dpo_lora_qwen05b_naive_v6
但 v6 不是默认推荐 checkpoint
继续训练前，先重新设计更宽的 prompt 7 curriculum 和回归保护
```

## 8. 继续 DPO 前的最小检查清单

| 检查项 | 为什么重要 |
|---|---|
| 起点 adapter 明确 | 从 SFT v3、DPO v6 还是其他 adapter 出发，含义不同 |
| DPO 数据不是同题背答案 | 要有改写和 held-out，避免只记固定问法 |
| eval_file 只看 preference | preference eval 不是最终行为验收 |
| batch 和 length 保守 | 8GB 显存下先保证能跑完 |
| output_dir 新名字 | 避免覆盖已有 artifact |
| 训练后立刻 compare + score | 不经过行为门禁，不接受 adapter |
| 写 report | 失败也要留下证据，失败是项目资产 |

## 9. 一个新 DPO 实验应该长什么样？

不要先改 `train_dpo.py`。通常先新建一份 config：

```text
configs/dpo_qwen05b_v9_my_probe.yaml
```

最小原则：

```text
sft_adapter_path: 从明确的可解释起点出发
dpo_file: 新的 preference train 数据
eval_file: 新的 preference eval 数据
output_dir: 新目录，不覆盖旧输出
learning_rate: 先保守
separate_ref_model: true
```

训练完以后，不要急着宣布成功。先回到 04：compare -> score -> report。

## 10. 这一页你要能讲什么

> 继续 DPO 的入口是 `scripts/train_dpo.py` 加一份 `configs/dpo_*.yaml`。数据格式是 `prompt/chosen/rejected`，训练时 policy model 是可训练的 base + adapter，reference model 是冻结对照。8GB 本地环境下要保守控制长度和 batch，并优先使用 separate reference。训练完成只得到一个候选 adapter，必须再经过固定 prompt 和 expanded behavior gate；本项目里 v6 是最好 DPO artifact，但因为 prompt 7 没过，所以默认推荐 checkpoint 仍然是 SFT v3。